In [6]:
import os
from pathlib import Path
import pandas as pd

# Define the root path for the lakehouse data
lakehouse_path = Path("data/lakehouse")

subfolders = ["day_ahead_prices", "total_load", "generation"]

# Function to recursively read all parquet files in a subfolder and combine them
def load_partitioned_parquet(subfolder_name):
    folder_path = lakehouse_path / subfolder_name
    if not folder_path.exists():
        print(f"Warning: Directory {folder_path} does not exist.")
        return None
    
    # Find all parquet files recursively (handles nested partitioning folders)
    parquet_files = list(folder_path.glob("**/*.parquet"))
    
    if not parquet_files:
        print(f"No parquet files found in {folder_path}")
        return None
    
    print(f"Found {len(parquet_files)} parquet files in '{subfolder_name}'. Reading and merging...")
    
    # Read and concatenate all parquet files
    df_list = [pd.read_parquet(file) for file in parquet_files]
    df = pd.concat(df_list, ignore_index=True)
    
    return df

# 1. Load each dataset
raw_data = {}
for sf in subfolders:
    raw_data[sf] = load_partitioned_parquet(sf)

# 2. Basic Inspection: Print shapes and first few rows to understand structure
for sf, df in raw_data.items():
    if df is not None:
        print(f"\n--- Dataset: {sf} ---")
        print(f"Shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        display(df.tail(3))







Found 83 parquet files in 'day_ahead_prices'. Reading and merging...
Found 76 parquet files in 'total_load'. Reading and merging...
Found 49 parquet files in 'generation'. Reading and merging...

--- Dataset: day_ahead_prices ---
Shape: (67927, 3)
Columns: ['timestamp', 'bidding_zone', 'price_eur_mwh']


,timestamp,bidding_zone,price_eur_mwh
67924,2026-07-29 22:30:00+00:00,DE_LU,142.66
67925,2026-07-29 22:45:00+00:00,DE_LU,142.20
67926,2026-07-29 23:00:00+00:00,DE_LU,144.05



--- Dataset: total_load ---
Shape: (165886, 5)
Columns: ['timestamp', 'bidding_zone', 'load_mw', 'total_load', 'total_load_mw']


,timestamp,bidding_zone,load_mw,total_load,total_load_mw
165883,2026-07-29 22:15:00+00:00,DE_LU,NaN,NaN,48316.44787
165884,2026-07-29 22:30:00+00:00,DE_LU,NaN,NaN,48212.45596
165885,2026-07-29 22:45:00+00:00,DE_LU,NaN,NaN,47983.32910



--- Dataset: generation ---
Shape: (125844, 25)
Columns: ['timestamp', 'bidding_zone', 'solar_mw', 'wind_onshore_mw', 'wind_offshore_mw', 'gas_mw', 'hard_coal_mw', 'nuclear_mw', 'Biomass_Actual Aggregated', 'Fossil Brown coal/Lignite_Actual Aggregated', 'Fossil Coal-derived gas_Actual Aggregated', 'Fossil Oil_Actual Aggregated', 'Geothermal_Actual Aggregated', 'Hydro Pumped Storage_Actual Aggregated', 'Hydro Pumped Storage_Actual Consumption', 'Hydro Run-of-river and poundage_Actual Aggregated', 'Hydro Water Reservoir_Actual Aggregated', 'Other_Actual Aggregated', 'Other renewable_Actual Aggregated', 'Waste_Actual Aggregated', 'Fossil Gas_Actual Aggregated', 'Fossil Hard coal_Actual Aggregated', 'Solar_Actual Aggregated', 'Wind Offshore_Actual Aggregated', 'Wind Onshore_Actual Aggregated']


,timestamp,bidding_zone,solar_mw,wind_onshore_mw,wind_offshore_mw,gas_mw,hard_coal_mw,nuclear_mw,Biomass_Actual Aggregated,Fossil Brown coal/Lignite_Actual Aggregated,...,Hydro Run-of-river and poundage_Actual Aggregated,Hydro Water Reservoir_Actual Aggregated,Other_Actual Aggregated,Other renewable_Actual Aggregated,Waste_Actual Aggregated,Fossil Gas_Actual Aggregated,Fossil Hard coal_Actual Aggregated,Solar_Actual Aggregated,Wind Offshore_Actual Aggregated,Wind Onshore_Actual Aggregated
125841,2026-07-29 23:15:00+02:00,DE_LU,NaN,NaN,NaN,NaN,NaN,NaN,4029.95993,10557.83054,...,1011.07298,178.0692,136.93158,72.096,795.6852,7204.95252,4669.78427,0.177,1110.268,8802.37967
125842,2026-07-29 23:30:00+02:00,DE_LU,NaN,NaN,NaN,NaN,NaN,NaN,3995.91903,10572.76095,...,1007.80568,71.9732,136.93158,72.096,799.6425,7249.33847,4648.02847,0.163,1139.286,8889.43368
125843,2026-07-29 23:45:00+02:00,DE_LU,NaN,NaN,NaN,NaN,NaN,NaN,3989.61503,10567.53271,...,1010.73347,34.3194,137.20358,72.096,752.4295,7147.19971,4635.93180,0.152,1128.798,8989.89897


In [7]:


import pandas as pd

def clean_and_merge_data(raw_data_dict, missing_threshold=0.90):
    print("Starting data cleaning and harmonization...")
    
    # --- 1. Clean Day-Ahead Prices ---
    df_price = raw_data_dict.get('day_ahead_prices')
    if df_price is not None:
        df_price['timestamp'] = pd.to_datetime(df_price['timestamp'], utc=True)
        df_price = df_price.sort_values('timestamp').drop_duplicates(subset=['timestamp'])
        # Keep bidding_zone if it exists
        price_cols = ['timestamp', 'price_eur_mwh']
        if 'bidding_zone' in df_price.columns:
            price_cols.append('bidding_zone')
        df_price = df_price[price_cols]

    # --- 2. Clean Total Load ---
    df_load = raw_data_dict.get('total_load')
    if df_load is not None:
        df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)
        
        # Consolidate load_mw and total_load using combine_first
        if 'load_mw' in df_load.columns and 'total_load' in df_load.columns:
            df_load['total_load_mw'] = df_load['load_mw'].combine_first(df_load['total_load'])
        elif 'load_mw' in df_load.columns:
            df_load['total_load_mw'] = df_load['load_mw']
        elif 'total_load' in df_load.columns:
            df_load['total_load_mw'] = df_load['total_load']
            
        load_cols = ['timestamp', 'total_load_mw']
        if 'bidding_zone' in df_load.columns:
            load_cols.append('bidding_zone')
        df_load = df_load[[c for c in load_cols if c in df_load.columns]]
        df_load = df_load.sort_values('timestamp').drop_duplicates(subset=['timestamp'])
        
        # Resample 15-min load to hourly mean safely without losing bidding_zone
        df_load.set_index('timestamp', inplace=True)
        numeric_cols_load = df_load.select_dtypes(include=['number']).columns
        
        # Resample numeric data, aggregate bidding zone by taking the first value
        agg_dict = {col: 'mean' for col in numeric_cols_load}
        if 'bidding_zone' in df_load.columns:
            agg_dict['bidding_zone'] = 'first'
            
        df_load = df_load.resample('h').agg(agg_dict).reset_index()

    # --- 3. Clean Generation ---
    df_gen = raw_data_dict.get('generation')
    if df_gen is not None:
        df_gen['timestamp'] = pd.to_datetime(df_gen['timestamp'], utc=True)
        
        mapping_rules = {
            'Solar_Actual Aggregated': 'solar_mw',
            'Wind Onshore_Actual Aggregated': 'wind_onshore_mw',
            'Wind Offshore_Actual Aggregated': 'wind_offshore_mw',
            'Fossil Gas_Actual Aggregated': 'gas_mw',
            'Fossil Hard coal_Actual Aggregated': 'hard_coal_mw',
            'Nuclear_Actual Aggregated': 'nuclear_mw'
        }
        
        for old_col, new_col in mapping_rules.items():
            if old_col in df_gen.columns and new_col in df_gen.columns:
                df_gen[new_col] = df_gen[new_col].combine_first(df_gen[old_col])
                df_gen.drop(columns=[old_col], inplace=True)
            elif old_col in df_gen.columns:
                df_gen.rename(columns={old_col: new_col}, inplace=True)

        df_gen = df_gen.sort_values('timestamp').drop_duplicates(subset=['timestamp'])
        
        # Drop columns that are mostly NaN (> 90% missing) before resampling
        nan_fraction = df_gen.isna().mean()
        cols_to_drop = nan_fraction[nan_fraction > missing_threshold].index
        cols_to_drop = [c for c in cols_to_drop if c not in ['timestamp', 'bidding_zone']]
        df_gen.drop(columns=cols_to_drop, inplace=True, errors='ignore')
        
        # Resample 15-min generation to hourly mean
        df_gen.set_index('timestamp', inplace=True)
        numeric_cols_gen = df_gen.select_dtypes(include=['number']).columns
        
        agg_dict_gen = {col: 'mean' for col in numeric_cols_gen}
        if 'bidding_zone' in df_gen.columns:
            agg_dict_gen['bidding_zone'] = 'first'
            
        df_gen = df_gen.resample('h').agg(agg_dict_gen).reset_index()

    # --- 4. Merge All Datasets on Timestamp ---
    print("Merging datasets...")
    master_df = df_price.copy()
    
    if df_load is not None:
        # Merge on timestamp (and bidding_zone if both have it)
        merge_keys = ['timestamp']
        if 'bidding_zone' in master_df.columns and 'bidding_zone' in df_load.columns:
            merge_keys.append('bidding_zone')
        master_df = pd.merge(master_df, df_load, on=merge_keys, how='inner')
        
    if df_gen is not None:
        merge_keys = ['timestamp']
        if 'bidding_zone' in master_df.columns and 'bidding_zone' in df_gen.columns:
            merge_keys.append('bidding_zone')
        master_df = pd.merge(master_df, df_gen, on=merge_keys, how='inner')
        
    master_df = master_df.sort_values('timestamp').reset_index(drop=True)
    
    print(f"Master DataFrame successfully created with shape: {master_df.shape}")
    return master_df

# Execute cleaning and merging function
master_energy_df = clean_and_merge_data(raw_data)
# display(master_energy_df.head())
# display(master_energy_df.info())

############################################ daily pipeline part##########################################################
master_energy_df.tail()

Starting data cleaning and harmonization...
Merging datasets...
Master DataFrame successfully created with shape: (31343, 10)


,timestamp,price_eur_mwh,bidding_zone,total_load_mw,solar_mw,wind_onshore_mw,wind_offshore_mw,gas_mw,hard_coal_mw,nuclear_mw
31338,2026-07-29 18:00:00+00:00,251.36,DE_LU,57070.340780,1709.42225,3378.069457,847.20200,7381.084000,4140.469552,NaN
31339,2026-07-29 19:00:00+00:00,254.43,DE_LU,56601.813995,35.91625,5256.842293,915.03975,7448.980010,4417.295112,NaN
31340,2026-07-29 20:00:00+00:00,206.99,DE_LU,53811.091160,0.22000,7692.424683,969.62475,7404.314720,4585.758975,NaN
31341,2026-07-29 21:00:00+00:00,179.05,DE_LU,51837.870900,0.16975,8824.529430,1098.91150,7210.560240,4663.436730,NaN
31342,2026-07-29 22:00:00+00:00,162.16,DE_LU,NaN,0.12675,8702.150960,1180.11625,6116.550172,4762.954325,NaN


In [8]:
import os
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
import pandas as pd

# Load environment variables from .env file to get the ENTSO-E API key
load_dotenv()
api_key = os.getenv("ENTSOE_API_KEY")
from pathlib import Path
import pandas as pd

def fetch_align_and_save_live_data(target_date_str, bidding_zone="DE_LU"):
    """
    1. Fetches raw prices, load, and generation data from ENTSO-E API.
    2. Aligns and formats everything to match the exact schema of master_energy_df (including bidding_zone).
    3. Saves them as Parquet partitions in their respective lakehouse folders.
    4. Returns the fully aligned live master chunk ready for feature engineering.
    """
    client = EntsoePandasClient(api_key=api_key)
    start = pd.Timestamp(target_date_str, tz='UTC')
    end = start + pd.Timedelta(days=1) - pd.Timedelta(hours=1)
    
    print(f"Fetching live ENTSO-E data for zone {bidding_zone} on {target_date_str}...")
    
    try:
        # --- 1. Fetch Raw Data from ENTSO-E ---
        prices = client.query_day_ahead_prices(bidding_zone, start=start, end=end)
        load = client.query_load(bidding_zone, start=start, end=end)
        generation = client.query_generation(bidding_zone, start=start, end=end, resolution='PT60M')
        
        # --- 2. Clean & Format Day-Ahead Prices ---
        df_price_live = pd.DataFrame({
            'timestamp': pd.to_datetime(prices.index, utc=True),
            'bidding_zone': bidding_zone,
            'price_eur_mwh': pd.to_numeric(prices.values.ravel(), errors='coerce')
        }).sort_values('timestamp').drop_duplicates(subset=['timestamp'])
        
        # --- 3. Clean & Format Total Load ---
        df_load_live = pd.DataFrame({
            'timestamp': pd.to_datetime(load.index, utc=True),
            'bidding_zone': bidding_zone,
            'total_load_mw': pd.to_numeric(load.values.ravel(), errors='coerce')
        }).sort_values('timestamp').drop_duplicates(subset=['timestamp'])
        
        # --- 4. Clean & Format Generation ---
        df_gen_live = generation.copy()
        if isinstance(df_gen_live.columns, pd.MultiIndex):
            df_gen_live.columns = ['_'.join(col).strip() for col in df_gen_live.columns.values]
            
        df_gen_live = df_gen_live.reset_index()
        if 'index' in df_gen_live.columns:
            df_gen_live.rename(columns={'index': 'timestamp'}, inplace=True)
        elif 'level_0' in df_gen_live.columns:
            df_gen_live.rename(columns={'level_0': 'timestamp'}, inplace=True)
            
        df_gen_live['timestamp'] = pd.to_datetime(df_gen_live['timestamp'], utc=True)
        
        # Ensure bidding_zone is explicitly included in generation data to match lakehouse schema
        df_gen_live['bidding_zone'] = bidding_zone
        
        # Map generation columns to match clean naming convention
        mapping_rules = {
            'Solar_Actual Aggregated': 'solar_mw',
            'Wind Onshore_Actual Aggregated': 'wind_onshore_mw',
            'Wind Offshore_Actual Aggregated': 'wind_offshore_mw',
            'Fossil Gas_Actual Aggregated': 'gas_mw',
            'Fossil Hard coal_Actual Aggregated': 'hard_coal_mw',
            'Nuclear_Actual Aggregated': 'nuclear_mw'
        }
        for old_col, new_col in mapping_rules.items():
            if old_col in df_gen_live.columns:
                df_gen_live.rename(columns={old_col: new_col}, inplace=True)
                
        df_gen_live = df_gen_live.sort_values('timestamp').drop_duplicates(subset=['timestamp'])

        # --- 5. Save to Respective Lakehouse Folders as Parquet ---
        year = start.year
        month = f"{start.month:02d}"
        
        
        lakehouse_root = Path("data/lakehouse")

        price_dir = (
                 lakehouse_root
                / "day_ahead_prices"
                / "zone=DE_LU"
                / f"year={year}"
                / f"month={month}")
        
        # Price partition
        price_dir.mkdir(parents=True, exist_ok=True)

# Save file with the date in the filename
        filename = f"price_{start.strftime('%Y-%m-%d')}.parquet"

        df_price_live.to_parquet(price_dir / filename, index=False)
        
        # Load partition
        load_dir = (
                         lakehouse_root
                        / "total_load"
                        / "zone=DE_LU"
                        / f"year={year}"
                        / f"month={month}")
                
                # Price partition
        load_dir.mkdir(parents=True, exist_ok=True)
        
        # Save file with the date in the filename
        filename = f"load_{start.strftime('%Y-%m-%d')}.parquet"
        
        df_load_live.to_parquet(load_dir / filename, index=False)
                

        
        # Generation partition
        gen_dir = (
                                 lakehouse_root
                                / "generation"
                                / "zone=DE_LU"
                                / f"year={year}"
                                / f"month={month}")
                        
                        # Price partition
        gen_dir.mkdir(parents=True, exist_ok=True)
                
                # Save file with the date in the filename
        filename = f"gen_{start.strftime('%Y-%m-%d')}.parquet"
                
        df_gen_live.to_parquet(gen_dir / filename, index=False)


        # gen_dir = lakehouse_root / "generation" / f"date={target_date_str}"
        # gen_dir.mkdir(parents=True, exist_ok=True)
        # df_gen_live.to_parquet(gen_dir / "actual_generation.parquet", index=False)
        
        print(f"Successfully saved aligned live partitions (Price, Load, Generation) for {target_date_str}.")
        
        # --- 6. Merge into a Live Master Chunk ---
        live_master_chunk = pd.merge(df_price_live, df_load_live, on=['timestamp', 'bidding_zone'], how='inner')
        
        # Merge generation safely using both timestamp and bidding_zone
        merge_gen_keys = ['timestamp']
        if 'bidding_zone' in df_gen_live.columns:
            merge_gen_keys.append('bidding_zone')
            
        live_master_chunk = pd.merge(live_master_chunk, df_gen_live, on=merge_gen_keys, how='inner')
        
        return live_master_chunk
        
    except Exception as e:
        print(f"Error during live ENTSO-E generation extraction: {e}")
        return None

# data = fetch_align_and_save_live_data("2026-07-31")

In [ ]:

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from dotenv import load_dotenv
from entsoe import EntsoePandasClient

# Your existing, already-working function — import it from wherever you
# defined it instead of redefining it here.
# from live_data import fetch_align_and_save_live_data

load_dotenv()
api_key = os.getenv("ENTSOE_API_KEY")


# ---------------------------------------------------------------------------
# 1. FORECAST DATA (fixed)
# ---------------------------------------------------------------------------

def fetch_next_day_forecast_inputs(target_date_str: str, bidding_zone: str = "DE_LU") -> pd.DataFrame:
    """
    Downloads tomorrow's day-ahead LOAD and WIND/SOLAR forecasts and shapes
    them so the output has the exact same column names as master_energy_df:

        timestamp, bidding_zone, total_load_mw, solar_mw,
        wind_onshore_mw, wind_offshore_mw, price_eur_mwh (NaN),
        gas_mw / hard_coal_mw / nuclear_mw (NaN — not forecastable)

    Matching the schema exactly is what lets this be pd.concat'ed straight
    onto master_updated and run through engineer_features() unchanged.
    """
    client = EntsoePandasClient(api_key=api_key)

    start = pd.Timestamp(target_date_str, tz="UTC")
    end = start + pd.Timedelta(days=1)

    print(f"Downloading forecasts for {target_date_str}...")

    load_forecast = client.query_load_forecast(bidding_zone, start=start, end=end)
    wind_solar = client.query_wind_and_solar_forecast(bidding_zone, start=start, end=end)

    # FIX: some entsoe-py versions return query_load_forecast as a
    # single-column DataFrame instead of a Series, which makes .values
    # 2-D (shape (n, 1)) -> "Per-column arrays must each be 1-dimensional".
    # np.ravel() flattens either shape safely to 1-D.
    load_values = np.ravel(load_forecast.values)


    df_load = load_forecast.reset_index().rename(columns={"index": "timestamp"})
    df_load["timestamp"] = pd.to_datetime(df_load["timestamp"], utc=True)
    df_load=df_load.rename(columns={"Forecasted Load":"total_load_mw"})



    df_gen = wind_solar.reset_index().rename(columns={"index": "timestamp"})
    df_gen["timestamp"] = pd.to_datetime(df_gen["timestamp"], utc=True)

    # FIX: rename ENTSO-E's raw column names to match master_energy_df
    df_gen = df_gen.rename(columns={
        "Solar": "solar_mw",
        "Wind Onshore": "wind_onshore_mw",
        "Wind Offshore": "wind_offshore_mw",
    })

    # FIX: wind/solar forecasts come back at 15-min resolution while
    # master_energy_df (and the load forecast) are hourly. Resampling to
    # hourly mean BEFORE merging avoids a messy outer join on mismatched
    # timestamps, which would otherwise produce more than 24 rows/day or
    # rows with partial NaNs. Same pattern your clean_and_merge_data
    # already uses for load/generation.
    df_load = (
        df_load.set_index("timestamp")
        .resample("h")
        .mean()
        .reset_index()
    )
    df_gen = (
        df_gen.set_index("timestamp")
        .resample("h")
        .mean()
        .reset_index()
    )

    result = pd.merge(df_load, df_gen, on="timestamp", how="outer")

    # FIX: small edge gaps (e.g. 1-2 hours) can appear after resampling
    # 15-min wind/solar data to hourly, usually from a UTC/local bidding-
    # zone boundary mismatch. These are raw INPUT features, not the
    # target, so interpolating a couple of missing hours from neighbours
    # is legitimate and introduces no leakage.
    forecast_input_cols = ["total_load_mw", "solar_mw", "wind_onshore_mw", "wind_offshore_mw"]
    forecast_input_cols = [c for c in forecast_input_cols if c in result.columns]
    result[forecast_input_cols] = result[forecast_input_cols].interpolate(limit_direction="both")

    # FIX: add bidding_zone + placeholder columns so the schema matches
    # master_updated exactly (needed for a clean concat downstream)
    result["bidding_zone"] = bidding_zone
    result["price_eur_mwh"] = np.nan
    for col in ["gas_mw", "hard_coal_mw", "nuclear_mw"]:
        result[col] = np.nan

    result = result.sort_values("timestamp").reset_index(drop=True)
    print(f"  -> {len(result)} forecast rows downloaded.")
    return result


# ---------------------------------------------------------------------------
# 2. FEATURE ENGINEERING 
# ---------------------------------------------------------------------------

LAG_HOURS = [24, 48, 168]
LAG_COLUMNS = ["price_eur_mwh", "total_load_mw", "residual_load_mw"]


def engineer_features(master_data: pd.DataFrame) -> pd.DataFrame:
    """
    One feature engineering function used for BOTH training rows (real
    price) and forecast rows (price_eur_mwh = NaN). Because lag/rolling
    features are built with .shift(), a value at time t only ever depends
    on strictly earlier timestamps -> no leakage, and forecast rows
    automatically get correct lag values from real history.
    """
    print("Starting feature engineering...")

    df = master_data.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)

    # -- 1. Calendar features -----------------------------------------
    df["hour"] = df.timestamp.dt.hour
    df["dayofweek"] = df.timestamp.dt.dayofweek
    df["month"] = df.timestamp.dt.month
    df["dayofyear"] = df.timestamp.dt.dayofyear
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df.month / 12)

    # -- 2. Residual load -----------------------------------------------
    # Uses only columns that ALSO exist as forecasts (total_load_mw,
    # solar_mw, wind_onshore_mw, wind_offshore_mw) so it's computable
    # identically for historical and forecast rows.
    df["residual_load_mw"] = (
        df["total_load_mw"] - df["solar_mw"] - df["wind_onshore_mw"] - df["wind_offshore_mw"]
    )

    # -- 3. Lag features --------------------------------------------------
    for col in LAG_COLUMNS:
        for lag in LAG_HOURS:
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)

    # -- 4. Rolling price stats (leak-free) --------------------------
    # shift(1) BEFORE rolling: the window ending at time t only contains
    # values strictly before t, so this is safe even when price at t
    # itself is NaN (a forecast row).
    df["price_rolling_mean_24"] = df["price_eur_mwh"].shift(1).rolling(24).mean()
    df["price_rolling_std_24"] = df["price_eur_mwh"].shift(1).rolling(24).std()

    # FIX: a sliding 24h window breaks down inside the forecast day itself.
    # Hour 1 of tomorrow has a window of 24 known past prices -> fine.
    # Hour 2's window now includes hour 1 of tomorrow, whose price is NaN
    # (not predicted yet) -> NaN. By hour 24, the whole window is NaN.
    # In real day-ahead forecasting you predict the whole next day at once,
    # BEFORE it starts, so "recent price behaviour" should be one snapshot
    # frozen at the last known actual hour, applied to every hour of the
    # forecast day — not a window that reaches into hours you haven't
    # predicted yet. ffill only ever copies a PAST value forward, so this
    # introduces no leakage.
    df["price_rolling_mean_24"] = df["price_rolling_mean_24"].ffill()
    df["price_rolling_std_24"] = df["price_rolling_std_24"].ffill()

    # -- 5. No fill here ----------------------------------------------
    # FIX: removed the old .bfill() over lag/rolling columns. It leaked
    # future values into early training rows and did nothing for
    # forecast rows (there's nothing after them to backfill from).
    # Remaining NaNs are handled explicitly downstream: dropped for
    # training, checked-and-raised for forecasting.

    return df


# ---------------------------------------------------------------------------
# 3. MODEL TRAINING (fixed)
# ---------------------------------------------------------------------------

MODEL_FEATURES = [
    "total_load_mw",
    "solar_mw",
    "wind_offshore_mw",
    "wind_onshore_mw",

    "hour",
    "dayofweek",
    "month",
    "dayofyear",
    "is_weekend",

    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",

    "residual_load_mw",

    "price_eur_mwh_lag_24",
    "price_eur_mwh_lag_48",
    "price_eur_mwh_lag_168",

    "total_load_mw_lag_24",
    "total_load_mw_lag_48",
    "total_load_mw_lag_168",

    "residual_load_mw_lag_24",
    "residual_load_mw_lag_48",
    "residual_load_mw_lag_168",

    "price_rolling_mean_24",
    "price_rolling_std_24",
]


def train_price_model(feature_df: pd.DataFrame) -> lgb.LGBMRegressor:
    # FIX: drop rows missing ANY required feature, not just price.
    # Without this, the first ~167 hours of history (no full lag_168 yet)
    # silently pass NaNs into LightGBM instead of being excluded.
    train_df = feature_df.dropna(subset=["price_eur_mwh"] + MODEL_FEATURES)

    if train_df.empty:
        raise ValueError(
            "No training rows survived feature engineering. "
            "Check that master_data spans at least 7+ days (needed for lag_168)."
        )

    X = train_df[MODEL_FEATURES]
    y = train_df["price_eur_mwh"]

    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        random_state=42,
        verbose=-1,
    )
    model.fit(X, y)

    print("Model trained:", X.shape)
    return model


# ---------------------------------------------------------------------------
# 4. DAILY PIPELINE (fixed)
# ---------------------------------------------------------------------------

def run_day_ahead_pipeline(master_data: pd.DataFrame, day_x: str):
    """
    Parameters
    ----------
    master_data : your existing master_energy_df
    day_x : "today" as a string, e.g. "2026-07-30" — the day whose
        actuals just became available.

    Returns
    -------
    (master_updated, model, forecast_result)
    """
    print(f"Processing {day_x}")

    # ---- STEP 1: download & append today's actuals ----------------------
    actual_day = fetch_align_and_save_live_data(day_x)

    master_updated = pd.concat([master_data, actual_day], ignore_index=True)
    master_updated = (
        master_updated
        # FIX: dedupe on (timestamp, bidding_zone), not timestamp alone,
        # and keep the newest copy of a re-run day.
        .drop_duplicates(subset=["timestamp", "bidding_zone"], keep="last")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    # ---- STEP 2: feature engineer + train on updated history -----------
    features = engineer_features(master_updated)
    model = train_price_model(features)

    # ---- STEP 3: download tomorrow's forecasts --------------------------
    next_day = (pd.Timestamp(day_x) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    # FIX: correct function name (was download_forecast_inputs, which
    # doesn't exist) and correct date (was day_x, should be next_day).
    forecast_inputs = fetch_next_day_forecast_inputs(next_day, "DE_LU")

    # ---- STEP 4: unified timeline -> one feature engineering call ------
    combined = pd.concat([master_updated, forecast_inputs], ignore_index=True)

    # DEFENSIVE: bidding_zone is constant per pipeline run, so any NaN
    # here (e.g. from a schema mismatch during concat) can be safely
    # forward-filled — this is metadata, not a model feature, and the
    # same fix cannot mask a genuine data problem in MODEL_FEATURES.
    combined["bidding_zone"] = combined["bidding_zone"].ffill().bfill()

    combined_features = engineer_features(combined)

    future_features = combined_features[
        combined_features.timestamp.dt.date == pd.Timestamp(next_day).date()
    ]


    print(future_features.info())


    # ---- Fail loudly instead of predicting on garbage --------------------
    if len(future_features) != 24:
        raise ValueError(
            f"Expected 24 forecast rows for {next_day}, got {len(future_features)}. "
            "Check that the ENTSO-E forecast query returned a full day."
        )

    nan_mask = future_features[MODEL_FEATURES].isna()
    if nan_mask.any().any():
        bad_cols = future_features[MODEL_FEATURES].columns[nan_mask.any()].tolist()
        raise ValueError(
            f"NaNs in forecast feature matrix for columns: {bad_cols}. "
            "Likely a forecast column-naming mismatch or insufficient history for lag_168."
        )

    # ---- STEP 5: predict --------------------------------------------------
    predictions = model.predict(future_features[MODEL_FEATURES])

    forecast_result = pd.DataFrame({
        "timestamp": future_features.timestamp.values,
        "forecast_price_eur_mwh": predictions,
    })

    return master_updated, model, forecast_result, future_features

In [22]:
a,b,c,d=run_day_ahead_pipeline(master_energy_df,"2026-07-31")

Processing 2026-07-31
Fetching live ENTSO-E data for zone DE_LU on 2026-07-31...
Successfully saved aligned live partitions (Price, Load, Generation) for 2026-07-31.
Starting feature engineering...
Model trained: (31177, 25)
  -> 22 forecast rows downloaded.
Starting feature engineering...
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 31405 to 31426
Data columns (total 43 columns):
 #   Column                                             Non-Null Count  Dtype              
---  ------                                             --------------  -----              
 0   timestamp                                          22 non-null     datetime64[us, UTC]
 1   price_eur_mwh                                      0 non-null      float64            
 2   bidding_zone                                       22 non-null     str                
 3   total_load_mw                                      22 non-null     float64            
 4   solar_mw                                           22

ValueError: Expected 24 forecast rows for 2026-08-01, got 22. Check that the ENTSO-E forecast query returned a full day.

In [18]:
d

,timestamp,price_eur_mwh,bidding_zone,total_load_mw,solar_mw,wind_onshore_mw,wind_offshore_mw,gas_mw,hard_coal_mw,nuclear_mw,...,price_eur_mwh_lag_48,price_eur_mwh_lag_168,total_load_mw_lag_24,total_load_mw_lag_48,total_load_mw_lag_168,residual_load_mw_lag_24,residual_load_mw_lag_48,residual_load_mw_lag_168,price_rolling_mean_24,price_rolling_std_24
31435,2026-07-31 00:00:00+00:00,NaN,DE_LU,43024.193690,0.000000,2871.604400,687.7775,NaN,NaN,NaN,...,4.56,156.05,62436.77373,60939.18754,45763.909540,40281.69777,-5220.05067,19085.361242,184.74125,18.65448
31436,2026-07-31 01:00:00+00:00,NaN,DE_LU,42584.790488,0.000000,2389.792777,477.9025,NaN,NaN,NaN,...,2.31,154.96,61587.97861,60561.61389,44326.501832,41468.36816,-5182.58102,16973.734085,184.74125,18.65448
31437,2026-07-31 02:00:00+00:00,NaN,DE_LU,43882.501677,0.000000,2051.214998,357.7725,NaN,NaN,NaN,...,1.38,145.35,60954.62460,59388.47982,42714.016735,42627.85101,-5821.51332,12046.160032,184.74125,18.65448
31438,2026-07-31 03:00:00+00:00,NaN,DE_LU,46980.385287,88.408280,1847.951165,334.7125,NaN,NaN,NaN,...,2.43,84.26,59804.31161,58387.44878,42859.591735,43138.10663,-6326.27343,10105.841585,184.74125,18.65448
31439,2026-07-31 04:00:00+00:00,NaN,DE_LU,52495.234849,2179.620685,1758.273080,357.0775,NaN,NaN,NaN,...,4.18,82.40,59372.12788,58391.49645,39500.753380,43744.91685,-6102.25313,7152.713228,184.74125,18.65448
31440,2026-07-31 05:00:00+00:00,NaN,DE_LU,57463.978620,8578.617640,1497.403775,371.6125,NaN,NaN,NaN,...,6.19,60.82,58945.38480,58534.02001,38644.379640,44430.24676,-4924.84795,7070.692010,184.74125,18.65448
31441,2026-07-31 06:00:00+00:00,NaN,DE_LU,60303.906144,18392.540693,1242.308630,375.7100,NaN,NaN,NaN,...,16.36,69.26,58605.87982,58078.56836,38858.313160,44746.99371,-4134.25645,8788.091282,184.74125,18.65448
31442,2026-07-31 07:00:00+00:00,NaN,DE_LU,60895.938277,28754.460778,974.104695,394.2525,NaN,NaN,NaN,...,32.96,66.38,58318.19178,57699.79152,39355.188375,45160.36432,-3232.56250,11304.576137,184.74125,18.65448
31443,2026-07-31 08:00:00+00:00,NaN,DE_LU,61228.580387,37557.304005,916.985005,410.7875,NaN,NaN,NaN,...,40.04,60.43,57717.11149,57349.65470,43313.919568,44791.38074,-2425.71679,15243.942018,184.74125,18.65448
31444,2026-07-31 09:00:00+00:00,NaN,DE_LU,60329.632316,43962.804460,1168.556685,452.9875,NaN,NaN,NaN,...,57.45,102.69,58108.32056,57410.94681,49233.521032,44985.82634,-1156.06811,20670.930317,184.74125,18.65448


In [20]:
d.info()


<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 31435 to 31458
Data columns (total 43 columns):
 #   Column                                             Non-Null Count  Dtype              
---  ------                                             --------------  -----              
 0   timestamp                                          24 non-null     datetime64[us, UTC]
 1   price_eur_mwh                                      0 non-null      float64            
 2   bidding_zone                                       24 non-null     str                
 3   total_load_mw                                      24 non-null     float64            
 4   solar_mw                                           24 non-null     float64            
 5   wind_onshore_mw                                    24 non-null     float64            
 6   wind_offshore_mw                                   24 non-null     float64            
 7   gas_mw                                             0 non-null      f

In [21]:
c

,timestamp,forecast_price_eur_mwh
0,2026-07-31 00:00:00,151.242736
1,2026-07-31 01:00:00,154.092514
2,2026-07-31 02:00:00,161.459466
3,2026-07-31 03:00:00,162.879007
4,2026-07-31 04:00:00,169.717783
5,2026-07-31 05:00:00,170.424395
6,2026-07-31 06:00:00,156.601227
7,2026-07-31 07:00:00,133.779628
8,2026-07-31 08:00:00,107.938783
9,2026-07-31 09:00:00,103.555897


In [116]:
df = pd.read_csv("DE_LU_clean_market.csv", encoding="cp1252")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 118: character maps to <undefined>